In [1]:
!pip install -q -U bitsandbytes accelerate peft

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import json, random, torch
from glob import glob
from PIL import Image
from datasets import Dataset
from transformers import (Qwen2VLForConditionalGeneration, AutoProcessor,
                          BitsAndBytesConfig, TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

DATA        = "/kaggle/input/datasets/zphilip/nougat-training-dataset-example"
MODEL       = "Qwen/Qwen2-VL-2B-Instruct"
OUT         = "/kaggle/working/qwen2vl-nougat"
PROMPT      = "Convert this document page to Markdown."
TRAIN_LIMIT = 500   
MAX_LEN     = 2048  

def load_pairs(root):
    rows = []
    for jp in sorted(glob(f"{root}/**/*.jsonl", recursive=True)):
        jdir = os.path.dirname(jp)
        with open(jp, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                d = json.loads(line)
                md  = d.get("markdown") or d.get("ground_truth") or d.get("text") or d.get("pretext")
                img = d.get("image") or d.get("image_path") or d.get("file_name")
                if not (img and md):
                    continue
                if not os.path.isabs(img):
                    cand = os.path.join(jdir, img)
                    if not os.path.exists(cand):
                        cand = os.path.join(jdir, "images", os.path.basename(img))
                    img = cand
                if os.path.exists(img):
                    rows.append({"image": img, "markdown": md})
    if rows:
        return rows
    for img in sorted(glob(f"{root}/**/*.png", recursive=True)):
        stem = os.path.splitext(img)[0]
        for ext in (".mmd", ".md", ".txt"):
            if os.path.exists(stem + ext):
                with open(stem + ext, encoding="utf-8") as f:
                    rows.append({"image": img, "markdown": f.read()})
                break
    return rows


data = load_pairs(DATA)
assert data, f"no pairs found under {DATA}"
random.seed(42); random.shuffle(data)
train = data[:int(0.8 * len(data))][:TRAIN_LIMIT]

print(f"total pairs : {len(data)}")
print(f"train pairs : {len(train)} (capped at {TRAIN_LIMIT})")
print(f"sample image: {train[0]['image']}")

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)

processor = AutoProcessor.from_pretrained(MODEL,
                                          min_pixels=128 * 28 * 28,
                                          max_pixels=384 * 28 * 28)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL, quantization_config=bnb, device_map={"": 0}, torch_dtype=torch.float16)
model = prepare_model_for_kbit_training(model)

model = get_peft_model(model, LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM"))
model.print_trainable_parameters()

IMG_TOK = processor.tokenizer.convert_tokens_to_ids("<|image_pad|>")
PAD_TOK = processor.tokenizer.pad_token_id

def collate(batch):
    texts, imgs = [], []
    for s in batch:
        try:
            img = Image.open(s["image"]).convert("RGB")
        except Exception:
            # corrupt / zero-byte file — fall back to a known-good sample
            s   = train[0]
            img = Image.open(s["image"]).convert("RGB")
        msgs = [
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": PROMPT}]},
            {"role": "assistant", "content": [
                {"type": "text", "text": s["markdown"]}]},
        ]
        texts.append(processor.apply_chat_template(msgs, tokenize=False))
        imgs.append(img)

    enc = processor(text=texts, images=imgs, padding=True, truncation=True,
                    max_length=MAX_LEN, return_tensors="pt")
    labels = enc["input_ids"].clone()
    labels[labels == PAD_TOK] = -100
    labels[labels == IMG_TOK] = -100
    enc["labels"] = labels
    return enc

args = TrainingArguments(
    output_dir=OUT,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=20,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to="none",
)

Trainer(model=model, args=args,
        train_dataset=Dataset.from_list(train),
        data_collator=collate).train()

model.save_pretrained(OUT)
processor.save_pretrained(OUT)
print(f"adapter saved to {OUT}")

total pairs : 14197
train pairs : 500 (capped at 500)
sample image: /kaggle/input/datasets/zphilip/nougat-training-dataset-example/0508/cond-mat0508359/10.png


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

trainable params: 2,179,072 || all params: 2,211,164,672 || trainable%: 0.0985


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
10,0.494868
20,0.459848
30,0.442956
40,0.367398
50,0.370445
60,0.334427
70,0.315853
80,0.339641
90,0.321895
100,0.286140


adapter saved to /kaggle/working/qwen2vl-nougat


In [6]:
from peft import PeftModel
import gradio as gr

DATA    = "/kaggle/input/datasets/zphilip/nougat-training-dataset-example"
BASE    = "Qwen/Qwen2-VL-2B-Instruct"
ADAPTER = "/kaggle/input/datasets/faiqahmad01/finetuned-qwen-adapter"
PROMPT  = "Convert this document page to Markdown."

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)

processor = AutoProcessor.from_pretrained(BASE,
                                          min_pixels=256 * 28 * 28,
                                          max_pixels=768 * 28 * 28)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
model = PeftModel.from_pretrained(model, ADAPTER)
model.eval()

@torch.no_grad()
def to_markdown(image, instruction=PROMPT, max_tokens=1024):
    msgs = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction or PROMPT}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc  = processor(text=[text], images=[image.convert("RGB")],
                     return_tensors="pt").to(model.device)
    out  = model.generate(**enc, max_new_tokens=int(max_tokens), do_sample=False)
    return processor.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                  skip_special_tokens=True)[0]

def load_pairs(root):
    rows = []
    for jp in sorted(glob(f"{root}/**/*.jsonl", recursive=True)):
        jdir = os.path.dirname(jp)
        with open(jp, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                d = json.loads(line)
                md  = d.get("markdown") or d.get("ground_truth") or d.get("text") or d.get("pretext")
                img = d.get("image") or d.get("image_path") or d.get("file_name")
                if not (img and md):
                    continue
                if not os.path.isabs(img):
                    cand = os.path.join(jdir, img)
                    if not os.path.exists(cand):
                        cand = os.path.join(jdir, "images", os.path.basename(img))
                    img = cand
                if os.path.exists(img):
                    rows.append({"image": img, "markdown": md})
    if rows:
        return rows
    for img in sorted(glob(f"{root}/**/*.png", recursive=True)):
        stem = os.path.splitext(img)[0]
        for ext in (".mmd", ".md", ".txt"):
            if os.path.exists(stem + ext):
                with open(stem + ext, encoding="utf-8") as f:
                    rows.append({"image": img, "markdown": f.read()})
                break
    return rows


data = load_pairs(DATA)
random.seed(42); random.shuffle(data)
cut = int(0.8 * len(data))
train, val = data[:cut], data[cut:]

def show(tag, samples):
    print(f"\n--- {tag} ---")
    for s in samples:
        print("\nfile:", s["image"])
        print("pred:", to_markdown(Image.open(s["image"]))[:400])
        print("gt  :", s["markdown"][:400])

show("3 TRAIN IMAGES", train[:3])
show("3 UNSEEN VAL IMAGES", val[:3])

def run(image, instruction, max_tokens):
    if image is None:
        return "", ""
    md = to_markdown(image, instruction, max_tokens)
    return md, md

with gr.Blocks(title="Qwen2-VL Nougat") as app:
    gr.Markdown("## Document → Markdown\nFine-tuned Qwen2-VL-2B + QLoRA on the Nougat dataset.")
    with gr.Row():
        with gr.Column():
            img    = gr.Image(type="pil", label="Page image")
            instr  = gr.Textbox(value=PROMPT, label="Instruction")
            tokens = gr.Slider(128, 2048, value=1024, step=64, label="Max new tokens")
            btn    = gr.Button("Convert", variant="primary")
        with gr.Column():
            raw     = gr.Textbox(label="Markdown (raw)", lines=20, show_copy_button=True)
            preview = gr.Markdown(label="Preview")
    btn.click(run, [img, instr, tokens], [raw, preview])

app.launch(share=True)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]


--- 3 TRAIN IMAGES ---

file: /kaggle/input/datasets/zphilip/nougat-training-dataset-example/0508/cond-mat0508359/10.png
pred: **Waves**, PhD Thesis, Delft University of Technology, (1997).

J. Muller Annales Geophysicae - Atmospheres Hydrospheres and Space Sciences **11**, (6): 525-531 (1993).

H. E. Stanley and P. Meakin, Nature **355**, 405 (1988).

G. Corso and L. S. Lucena, Physica A (to appear) (2005).
gt  : Waves._ PhD Thesis, Delft University of Technology, (1997).
* (13) J. Muller Annales Geophysicae - Atmospheres Hydrospheres and Space Sciences **11**, (6): 525-531 (1993).
* (14) H. E. Stanley and P. Meakin, Nature **355**, 405 (1988).
* (15) G. Corso and L. S. Lucena, Physica A (to appear) (2005).

file: /kaggle/input/datasets/zphilip/nougat-training-dataset-example/0508/hep-ex0508059/05.png
pred: "sea" are arise. This picture is qualitatively consistent with quantum-chromodynamic ideas. Therefore when we speak of the complex quark or quark-glue structure of cryptoexotic ba